In [5]:
import pandas as pd
import numpy as np

# 1. โหลดข้อมูลดิบจากไฟล์ CSV (เปลี่ยนชื่อไฟล์ให้ตรงกับของคุณ)
input_filename = '/content/iot_sensor_raw_data_extended.csv'
df = pd.read_csv(input_filename)

print("--- ข้อมูลก่อนทำความสะอาด (5 แถวแรก) ---")
print(df.head())
initial_row_count = len(df)

# ---------------------------------------------
# 2. เริ่มทำความสะอาดข้อมูลตาม Data Quality Rules
# ---------------------------------------------
df_cleaned = df.copy()

# ลบข้อมูลซ้ำ: ห้ามมีข้อมูลซ้ำที่มี device_id และ timestamp เดียวกัน
df_cleaned = df_cleaned.drop_duplicates(subset=['device_id', 'timestamp'], keep='first')

# จัดการ Missing Value: device_id ห้ามเป็นค่าว่าง
df_cleaned = df_cleaned.dropna(subset=['device_id'])

# แปลงค่าผิดปกติ (Outlier) เป็น NULL
# กฎข้อ 1: อุณหภูมิต้องอยู่ระหว่าง 0–50 องศาเซลเซียส
if 'temperature' in df_cleaned.columns:
    df_cleaned.loc[(df_cleaned['temperature'] < 0) | (df_cleaned['temperature'] > 50), 'temperature'] = np.nan

# กฎข้อ 2: ค่าแบตเตอรี่ต้องอยู่ระหว่าง 0–100
if 'battery' in df_cleaned.columns:
    df_cleaned.loc[(df_cleaned['battery'] < 0) | (df_cleaned['battery'] > 100), 'battery'] = np.nan

# เพิ่มคอลัมน์ data_quality_status
# หากคอลัมน์ใดในแถวนั้นมีค่าเป็น NaN (เช่น ข้อมูลหาย หรือเป็น Outlier ที่ถูกแปลงเป็น NULL) ให้สถานะเป็น Needs Review
df_cleaned['data_quality_status'] = df_cleaned.apply(
    lambda row: 'Needs Review' if row.isnull().any() else 'Passed',
    axis=1
)

final_row_count = len(df_cleaned)

print("\n--- ข้อมูลหลังทำความสะอาด (5 แถวแรก) ---")
print(df_cleaned.head())

# ---------------------------------------------
# 3. เปรียบเทียบจำนวนแถวก่อนและหลังทำความสะอาด
# ---------------------------------------------
print("\n--- สรุปการเปรียบเทียบ ---")
print(f"จำนวนแถวก่อนทำความสะอาด: {initial_row_count} แถว")
print(f"จำนวนแถวหลังทำความสะอาด: {final_row_count} แถว")

# 4. บันทึกผลลัพธ์เป็นไฟล์ Cleaned Data
output_filename = 'cleaned_sensor_data.csv'
df_cleaned.to_csv(output_filename, index=False)
print(f"\nบันทึกไฟล์ข้อมูลที่ทำความสะอาดแล้วไปที่: {output_filename}")

--- ข้อมูลก่อนทำความสะอาด (5 แถวแรก) ---
             timestamp device_id  temp_c motion  battery
0  2026-07-12 14:50:00   SN-9982    28.3  false     88.0
1  2026-07-12 14:50:00   SN-9983    28.2   true     94.0
2  2026-07-12 14:50:00   SN-9984    29.7   true     76.0
3  2026-07-12 14:50:00   SN-9985    27.8   true     81.0
4  2026-07-12 14:50:00   SN-9986    30.7   true     69.0

--- ข้อมูลหลังทำความสะอาด (5 แถวแรก) ---
             timestamp device_id  temp_c motion  battery data_quality_status
0  2026-07-12 14:50:00   SN-9982    28.3  false     88.0              Passed
1  2026-07-12 14:50:00   SN-9983    28.2   true     94.0              Passed
2  2026-07-12 14:50:00   SN-9984    29.7   true     76.0              Passed
3  2026-07-12 14:50:00   SN-9985    27.8   true     81.0              Passed
4  2026-07-12 14:50:00   SN-9986    30.7   true     69.0              Passed

--- สรุปการเปรียบเทียบ ---
จำนวนแถวก่อนทำความสะอาด: 480 แถว
จำนวนแถวหลังทำความสะอาด: 453 แถว

บันทึกไฟล์ข้อมูลที